# Main analysis notebook

In [ ]:
import gc
from pathlib import Path

import cyanomembranes as cm

import sys
from pathlib import Path
import time

# Add the folder that CONTAINS membrane_analysis/ to the path
sys.path.insert(0, str(Path("..").resolve()))  # adjust if needed
from membrane_analysis.config import (
    AnalysisType,
    make_exp_config,
)
from membrane_analysis.pipeline import run_scenario
from membrane_analysis.plotting import plot_diffusion, plot_fpt, plot_rate
from scenarios import ALL_SCENARIOS

### Create output directories; determine cores

In [ ]:
OUT = Path("../output")
DATA_DIR = Path("../data_cyano")
N_PROCESSES = 10
OUT.mkdir(exist_ok=True)

### Make protein shadows

In [ ]:
proteins    = cm.pdb_utils.process_proteins(
    DATA_DIR, OUT / "protein_shadows",
    ["4H13-cytb6f.trpdb", "3WU2-PSII-ThermosynVul.pdb"]
)
cytb6f_area = proteins["4H13-cytb6f"]["polygon"][0].area
psii_area   = proteins["3WU2-PSII-ThermosynVul"]["polygon"][0].area


### Perform analyses as defined in notebooks/scenario.py and membrane_analysis/config.py

Here notebook/scenario.py defines the overall experiment and membrane_analysis/config describes how each of the available analysis types is performed. The run_scenario function is the main function of the pipeline (see membrane_analysis/pipeline.py)

In [ ]:
for scenario in ALL_SCENARIOS:
    print(f"\n{'='*60}")
    print(f"Running: {scenario.name} - {scenario.analysis_type.name}")
    print(f"\n{'='*60}")

    exp_cfg = make_exp_config(
        scenario=scenario,
        cytb6f_area=cytb6f_area,
        psii_area=psii_area,
        n_processes=N_PROCESSES,
    )

    exp_cfg.store_history = True

    #run_scenario(scenario, exp_cfg, OUT)

   # time.sleep(30*60)

print("\n All simulations done")

In [ ]:
exp_cfg

In [ ]:
PLOT_DISPATCH = {
    AnalysisType.DIFFUSION: plot_diffusion,
    AnalysisType.FPT: plot_fpt,
    AnalysisType.RATE: plot_rate
    #AnalysisType.AIM: plot_aim
}

for scenario in ALL_SCENARIOS:
    try:
        plot_fn = PLOT_DISPATCH[scenario.analysis_type]
        plot_fn(scenario, OUT)
    except:
        print("didn't work")